In [1]:
import duckdb
import pandas as pd
from scraper import AnimeScraperFormatterDF
import time
from tqdm import tqdm
from db.update import *

c:\Users\gonza\Documents\Projects\ds-log-2026\w04\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
anime_conn = duckdb.connect(database = "data/anime.db")

# Open and read the schema file
with open("db/schema.sql", "r") as f:
    schema_sql = f.read()

anime_conn.execute(schema_sql)

In [4]:

def get_all_anime_info(anime_id):
    anime_client = AnimeScraperFormatterDF(anime_id)
    # Get info:
    if anime_conn.execute(f"select * from show_info where id = {anime_id}").df().empty or\
       anime_conn.execute(f"select * from show_genres where show_id = {anime_id}").df().empty or\
       anime_conn.execute(f"select * from show_themes where show_id = {anime_id}").df().empty or\
       anime_conn.execute(f"select * from show_relationships where source_id = {anime_id}").df().empty:
        info, genres, themes, relations  = anime_client.get_full_payload()
        upsert_show_info(anime_conn, info, genres, themes, relations)
    # Get episodes:
    if anime_conn.execute(f"select * from anime_episodes where anime_id = {anime_id}").df().empty:
        episodes = anime_client.get_episodes()
        upsert_episodes(anime_conn, episodes)
    # Get reviews:
    if anime_conn.execute(f"select * from reviews where anime_id = {anime_id}").df().empty:
        reviews, reactions, _ = anime_client.get_reviews(preliminary = True, spoilers = True)
        upsert_reviews(anime_conn, reviews, reactions)
    # Get forum messages:
    missing_forum_episodes = anime_conn.execute("""
                    select e.*
                    from anime_episodes as e
                    left join forum_messages as f on e.forum_topic_id = f.forum_id
                    where f.forum_id is null
                    """).df()["forum_topic_id"].tolist()
    for forum_topic_id in tqdm(missing_forum_episodes, desc="Fetching forum messages"):
        forum_messages = anime_client.get_forum_messages(forum_topic_id)
        upsert_forum_messages(anime_conn, forum_messages)
        time.sleep(1)

    # Get info on related shows and retrieve all its info as well:
    missing_related_shows = anime_conn.execute("""
                   select r.*
                   from show_relationships as r
                   left join show_info as s on r.id = s.id
                   where r.type = 'anime' and s.title is null
                   """).df()["id"].tolist()
    for related_show_id in tqdm(missing_related_shows, desc="Fetching related shows"):
        get_all_anime_info(related_show_id)
        time.sleep(5)

anime_id = 54492
get_all_anime_info(anime_id)

Fetching forum messages: 100%|██████████| 24/24 [00:51<00:00,  2.14s/it]
Fetching forum messages: 0it [00:00, ?it/s]1 [00:00<?, ?it/s]
Fetching related shows: 0it [00:00, ?it/s]
Fetching related shows: 100%|██████████| 1/1 [00:08<00:00,  8.65s/it]
